##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# -----------------------------
# 1) Load CIFAR-10
# -----------------------------
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]

# Keep labels as integers (SparseCategoricalCrossentropy)
y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")

# Convert images to float32
x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")

# -----------------------------
# 2) Data augmentation
# -----------------------------
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")

#Build MobileNetV2
mobilenet_base = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

mobilenet_base.trainable = False  # freeze backbone

mobilenet_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Resizing(224, 224, interpolation="bilinear"),
    layers.Lambda(preprocess_input),   # IMPORTANT for MobileNetV2
    mobilenet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10)   # logits
], name="cifar10_mobilenetv2")

mobilenet_model.summary()

# Compile + Train
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1),
]

history = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

test_loss_m, test_acc_m = mobilenet_model.evaluate(x_test, y_test, verbose=0)
print("MobileNetV2 (frozen) test accuracy:", test_acc_m)
print("MobileNetV2 (frozen) test loss:", test_loss_m)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step



Model: "cifar10_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 695s 981ms/step - accuracy: 0.6808 - loss: 0.9211 - val_accuracy: 0.8062 - val_loss: 0.5565 - learning_rate: 0.0010
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 718s 1s/step - accuracy: 0.7466 - loss: 0.7278 - val_accuracy: 0.8324 - val_loss: 0.4926 - learning_rate: 0.0010
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 733s 1s/step - accuracy: 0.7578 - loss: 0.6930 - val_accuracy: 0.8218 - val_loss: 0.5207 - learning_rate: 0.0010
MobileNetV2 (frozen) test accuracy: 0.8245000243186951
MobileNetV2 (frozen) test loss: 0.5134179592132568


In [3]:
#Fine-Tune
mobilenet_base.trainable = True

for layer in mobilenet_base.layers[:-30]:
    layer.trainable = False

print("Trainable layers in backbone:",
      sum(l.trainable for l in mobilenet_base.layers),
      "/", len(mobilenet_base.layers))

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_ft = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

test_loss_m_ft, test_acc_m_ft = mobilenet_model.evaluate(x_test, y_test, verbose=0)
print("MobileNetV2 (fine-tuned) test accuracy:", test_acc_m_ft)
print("MobileNetV2 (fine-tuned) test loss:", test_loss_m_ft)

Trainable layers in backbone: 30 / 154
Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 944s 1s/step - accuracy: 0.7143 - loss: 0.8274 - val_accuracy: 0.8304 - val_loss: 0.5076
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 923s 1s/step - accuracy: 0.7760 - loss: 0.6451 - val_accuracy: 0.8406 - val_loss: 0.4590
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 909s 1s/step - accuracy: 0.7990 - loss: 0.5751 - val_accuracy: 0.8554 - val_loss: 0.4071
MobileNetV2 (fine-tuned) test accuracy: 0.8575999736785889
MobileNetV2 (fine-tuned) test loss: 0.4221031367778778


In [4]:
# Compare with ResNet + Custom CNN

custom_cnn_acc = 0.7028
resnet_ft_acc = 0.9162

results = {
    "Custom CNN test acc": custom_cnn_acc,
    "ResNet50V2 fine-tuned test acc": resnet_ft_acc,
    "MobileNetV2 fine-tuned test acc": test_acc_m_ft,
}

for k, v in results.items():
    print(f"{k}: {v:.4f}")

Custom CNN test acc: 0.7028
ResNet50V2 fine-tuned test acc: 0.9162
MobileNetV2 fine-tuned test acc: 0.8576


1) Which model achieved the highest accuracy?

The fine-tuned ResNet50V2 achieved the highest accuracy

2) Which model trained faster?

The custom CNN trained much faster

3) How does architecture explain the differences?

Custom CNN:
Shallow network with limited feature extraction capacity → struggles on CIFAR-10 → lower accuracy.

MobileNetV2:
Lightweight architecture using depthwise separable convolutions → efficient and strong pretrained features → good accuracy.

ResNet50V2:
Deep residual architecture → strong hierarchical feature learning → best transfer learning performance.

The pretrained models benefit from ImageNet feature representations, while the custom CNN learns from scratch, which explains the large performance gap.